# SAM segmentation cho FLIR — chạy trên Kaggle

Sinh seg map cho từng ảnh RGB của FLIR bằng SAM (**ViT-L**), **gộp tất cả mask thành
1 ảnh nhị phân** (vùng đối tượng trắng, nền đen) — tương đương kết quả của
`SAM + process_masks.py` nhưng không tạo hàng nghìn file trung gian.

## Cách dùng
1. **Cell 1** (Cài đặt + tải checkpoint) — chạy 1 lần.
2. **Cell 2** (Cấu hình) — sửa nếu cần: `INPUT_ALIGN`, `FILENAME_FILTER`, `TEST_LIMIT`.
3. **Cell 3** (Chạy) — sinh seg, nén thành `flir_seg.zip`.
4. Tải `flir_seg.zip` từ tab **Output** của notebook → đó là dataset mới của bạn
   (folder `seg/` giữ nguyên tên ảnh gốc, chỉ đổi đuôi sang `.png`).

> Lần đầu chạy nên đặt `TEST_LIMIT = 3` ở Cell 2 để kiểm tra trước.
> Kiểm tra dòng `white_ratio` cuối cùng: nên nằm trong ~0.05–0.5.

In [ ]:
# ============ CELL 1: CÀI ĐẶT + TẢI CHECKPOINT (chạy 1 lần) ============
!pip install -q 'git+https://github.com/facebookresearch/segment-anything.git'
!pip install -q opencv-python pycocotools matplotlib

import os
CKPT = "/kaggle/working/sam_vit_l_0b3195.pth"
if not (os.path.isfile(CKPT) and os.path.getsize(CKPT) > 1_000_000_000):
    !wget -q -O {CKPT} https://dl.fbaipublicfiles.com/segment_anything/sam_vit_l_0b3195.pth
print("Checkpoint sẵn sàng:", CKPT, os.path.getsize(CKPT) // 1_000_000, "MB")

In [ ]:
# ============ CELL 2: CẤU HÌNH — sửa tại đây ============
INPUT_ALIGN = None        # None = tự dò; hoặc ghi rõ, vd "/kaggle/input/flir/align"
FILENAME_FILTER = ""      # "" = tất cả; "FLIR_00" = chỉ ảnh chứa chuỗi này (chia phiên)
TEST_LIMIT = 0            # 3 = chạy thử 3 ảnh đầu; 0 = chạy hết
RESUME = True             # bỏ qua ảnh đã có seg
BASE_OUT = "/kaggle/working"
SEG_FOLDER = "seg"

MASK_GEN_PARAMS = dict(
    points_per_side=32,   # 16 = nhanh gấp ~2
    pred_iou_thresh=0.88,
    stability_score_thresh=0.95,
    crop_n_layers=0,
    crop_n_points_downscale_factor=1,
    min_mask_region_area=100,
)

In [ ]:
# ============ CELL 3: CHẠY — sinh seg cho toàn bộ ảnh RGB ============
import glob, json, os, shutil, time
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
import torch

from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

CKPT = "/kaggle/working/sam_vit_l_0b3195.pth"   # khớp checkpoint ở Cell 1
SAM_MODEL_TYPE = "vit_l"

# ---- 1) Tìm folder align chứa JPEGImages ----
def find_align_root():
    if INPUT_ALIGN:
        assert os.path.isdir(os.path.join(INPUT_ALIGN, "JPEGImages")), INPUT_ALIGN
        return INPUT_ALIGN
    for cand in sorted(glob.glob("/kaggle/input/*/align")) + sorted(glob.glob("/kaggle/input/*")):
        if os.path.isdir(os.path.join(cand, "JPEGImages")):
            return cand
    raise SystemExit("Không tìm thấy folder align. Hãy set INPUT_ALIGN ở Cell 2.")

def discover_rgb_images(align_root):
    jpeg = os.path.join(align_root, "JPEGImages")
    rgb = []
    for pat in ("*_RGB.jpg", "*_RGB.jpeg", "*_RGB.png", "*_RGB.JPG"):
        rgb += glob.glob(os.path.join(jpeg, pat))
    rgb = sorted(set(rgb))
    if not rgb:
        rdir = os.path.join(align_root, "RGB")
        if os.path.isdir(rdir):
            rgb = sorted(glob.glob(os.path.join(rdir, "*.jpg")) +
                         glob.glob(os.path.join(rdir, "*.jpeg")) +
                         glob.glob(os.path.join(rdir, "*.png")))
    if not rgb:
        raise SystemExit(f"Không thấy ảnh RGB trong {jpeg}. 10 file đầu: {os.listdir(jpeg)[:10]}")
    if FILENAME_FILTER:
        rgb = [p for p in rgb if FILENAME_FILTER in os.path.basename(p)]
    return rgb

align_root = find_align_root()
rgb_images = discover_rgb_images(align_root)
print(f">> align root : {align_root}")
print(f">> số ảnh RGB : {len(rgb_images)}  (filter={FILENAME_FILTER or 'all'})")

# ---- 2) Load SAM ----
sam = sam_model_registry[SAM_MODEL_TYPE](checkpoint=CKPT)
sam.to("cuda"); sam.eval()
mask_generator = SamAutomaticMaskGenerator(sam, **MASK_GEN_PARAMS)

# ---- 3) Sinh mask + gộp -> seg map nhị phân (giống SAM + process_masks.py) ----
SEG_DIR = os.path.join(BASE_OUT, SEG_FOLDER)
os.makedirs(SEG_DIR, exist_ok=True)
manifest, failures = [], []
done = skipped = 0
start = time.time()

for rgb_path in tqdm(rgb_images, desc="SAM", unit="img"):
    stem = os.path.splitext(os.path.basename(rgb_path))[0]
    seg_path = os.path.join(SEG_DIR, stem + ".png")
    if RESUME and os.path.isfile(seg_path) and os.path.getsize(seg_path) > 0:
        skipped += 1
        manifest.append([os.path.basename(rgb_path), stem + ".png"])
        continue
    if TEST_LIMIT and done >= TEST_LIMIT:
        break
    try:
        image = np.array(Image.open(rgb_path).convert("RGB"))
        with torch.inference_mode():
            masks = mask_generator.generate(image)
        merged = np.zeros(image.shape[:2], bool)
        for m in masks:
            merged |= m["segmentation"]
        seg_img = np.zeros((*image.shape[:2], 3), np.uint8)
        seg_img[merged] = 255
        Image.fromarray(seg_img).save(seg_path)
        done += 1
        manifest.append([os.path.basename(rgb_path), stem + ".png"])
    except Exception as e:
        failures.append([os.path.basename(rgb_path), str(e)])
        print("!!", os.path.basename(rgb_path), e)
    if done in (1, 5, 20) or (done and done % 200 == 0):
        el = time.time() - start
        print(f"... {done} xong, ~{el/done:.2f}s/ảnh, ETA ~{el/done*(len(rgb_images)-skipped-done)/60:.1f} phút")

print(f"\n>> Xong: processed={done}, skipped={skipped}, failed={len(failures)} trong {time.time()-start:.0f}s")

# ---- 4) Nén dataset + báo cáo ----
with open(os.path.join(BASE_OUT, "manifest.json"), "w") as f:
    json.dump(manifest, f, indent=1)
with open(os.path.join(BASE_OUT, "failures.json"), "w") as f:
    json.dump(failures, f, indent=1)

zip_path = shutil.make_archive(os.path.join(BASE_OUT, "flir_seg"), "zip",
                               root_dir=BASE_OUT, base_dir=SEG_FOLDER)
print(f">> Dataset mới : {SEG_DIR} ({len(manifest)} seg maps)")
print(f">> Zip         : {zip_path} ({os.path.getsize(zip_path)/1e6:.1f} MB)")

for s in sorted(glob.glob(os.path.join(SEG_DIR, "*.png")))[:5]:
    a = np.array(Image.open(s))
    print(f"   {os.path.basename(s)}  size={a.shape[:2]}  white_ratio={np.mean(a > 0):.3f}")
if failures:
    print(f">> {len(failures)} lỗi trong failures.json — đầu tiên: {failures[0]}")